# Healthcare Supply in NRW: Doctors and Hospital Beds by District

**Author:** Yuliia Butenko

**Question:** Where in North Rhine-Westphalia is the supply of general practitioners (Hausärzte), specialists and hospital beds lowest, how has it changed since 2015, and which districts are at risk?

**Data:** INKAR, BBSR Bonn (Laufende Raumbeobachtung, Ausgabe 07/2025), 53 districts (Kreise and kreisfreie Städte) of NRW.

| Indicator | Unit | Years |
|---|---|---|
| Doctors, GPs (Hausärzte), general practitioners, internists | per 10,000 residents | 2015–2024 |
| Pediatricians | per 10,000 children under 15 | 2015–2024 |
| Hospital beds | per 1,000 residents | 2004–2022 |
| Population | residents | 1995–2023 |

**Steps:** SQL in BigQuery (see `sql/`) → this notebook (trends, statistical tests, risk zones) → Tableau map.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

URL = 'https://raw.githubusercontent.com/ButenkoYuliia/nrw-healthcare-supply/main/data/nrw_healthcare_long.csv'
long = pd.read_csv(URL, dtype={'ags': str})
print(long.shape)
long.head()

## 2. Wide table: one row per district and year

Same logic as `sql/01_build_wide_table.sql`.

In [ ]:
names = {
    'doctors_per_10000': 'doctors',
    'gps_per_10000': 'gps',
    'general_practitioners_per_10000': 'general_practitioners',
    'internists_per_10000': 'internists',
    'pediatricians_per_10000_children': 'pediatricians',
    'hospital_beds_per_1000': 'hospital_beds',
    'population': 'population',
}
wide = (long.assign(indicator=long['indicator'].map(names))
            .pivot_table(index=['ags', 'kreis', 'kreis_type', 'year'],
                         columns='indicator', values='value')
            .reset_index())
wide.columns.name = None
print(wide['kreis'].nunique(), 'districts,', wide['year'].min(), '-', wide['year'].max())
wide[wide['year'] == 2022].describe().round(2)

## 3. Trends 2015–2024: urban vs rural

Urban = kreisfreie Städte (and Städteregion Aachen), rural = Kreise.

In [ ]:
sns.set_theme(style='whitegrid')
colors = {'urban': '#8FA67A', 'rural': '#B7A48E'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col, title in zip(axes,
                          ['gps', 'doctors', 'hospital_beds'],
                          ['GPs per 10,000', 'All doctors per 10,000', 'Hospital beds per 1,000']):
    d = wide.dropna(subset=[col])
    d = d[d['year'] >= (2004 if col == 'hospital_beds' else 2015)]
    t = d.groupby(['year', 'kreis_type'])[col].mean().unstack()
    for k in ['urban', 'rural']:
        ax.plot(t.index, t[k], marker='o', ms=4, color=colors[k], label=k, lw=2)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.legend()
plt.suptitle('Average supply per district, NRW', fontsize=13)
plt.tight_layout()
plt.show()

## 4. GP supply by district, 2024

In [ ]:
gp = (wide.pivot_table(index=['kreis', 'kreis_type'], columns='year', values='gps')[[2015, 2024]]
          .reset_index()
          .rename(columns={2015: 'gps_2015', 2024: 'gps_2024'}))
gp['change_pct'] = ((gp['gps_2024'] - gp['gps_2015']) / gp['gps_2015'] * 100).round(1)
median_gps = gp['gps_2024'].median()

g = gp.sort_values('gps_2024')
fig, ax = plt.subplots(figsize=(9, 13))
ax.barh(g['kreis'], g['gps_2024'], color=g['kreis_type'].map(colors))
ax.axvline(median_gps, color='#5A4A3A', ls='--', lw=1)
ax.text(median_gps + 0.05, 0, f'NRW median {median_gps:.2f}', va='center', color='#5A4A3A')
ax.set_xlabel('GPs per 10,000 residents (2024)')
ax.set_title('GP supply by district, NRW 2024 (olive = urban, beige = rural)')
plt.tight_layout()
plt.show()

## 5. Statistical tests

**5.1 Urban vs rural.** Are the differences between urban and rural districts statistically significant? With only 22 urban and 31 rural districts and non-normal distributions, the **Mann-Whitney U test** is used (α = 0.05).

In [ ]:
rows = []
for col, year in [('gps', 2024), ('doctors', 2024), ('pediatricians', 2024),
                  ('internists', 2024), ('hospital_beds', 2022)]:
    d = wide[wide['year'] == year]
    u = d.loc[d['kreis_type'] == 'urban', col]
    r = d.loc[d['kreis_type'] == 'rural', col]
    stat, p = stats.mannwhitneyu(u, r, alternative='two-sided')
    rows.append({'indicator': col, 'year': year,
                 'median_urban': round(u.median(), 2), 'median_rural': round(r.median(), 2),
                 'p_value': round(p, 4), 'significant': p < 0.05})
pd.DataFrame(rows)

**5.2 Has GP supply declined?** Paired comparison of the same 53 districts in 2015 and 2024 with the **Wilcoxon signed-rank test**.

In [ ]:
stat, p = stats.wilcoxon(gp['gps_2024'], gp['gps_2015'])
print(f'Districts with a decline: {(gp.change_pct < 0).sum()} of {len(gp)}')
print(f'Median change: {gp.change_pct.median():.1f} %')
print(f'Wilcoxon p-value: {p:.4f}')

## 6. Correlations between indicators (2022)

In [ ]:
cols = ['gps', 'general_practitioners', 'internists', 'pediatricians', 'doctors', 'hospital_beds']
corr = wide[wide['year'] == 2022][cols].corr().round(2)
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, cmap='BrBG', vmin=-1, vmax=1, square=True)
plt.title('Correlation between supply indicators, 53 districts, 2022')
plt.tight_layout()
plt.show()

## 7. Risk zones

A district is a **risk zone** when its GP supply in 2024 is below the NRW median **and** lower than in 2015 (same logic as `sql/04_risk_zones.sql`).

In [ ]:
pop = wide[wide['year'] == 2023].set_index('kreis')['population']
gp['population_2023'] = gp['kreis'].map(pop).astype(int)
gp['risk_zone'] = (gp['gps_2024'] < median_gps) & (gp['gps_2024'] < gp['gps_2015'])

risk = gp[gp['risk_zone']].sort_values('change_pct')
share = risk['population_2023'].sum() / gp['population_2023'].sum() * 100
print(f'{len(risk)} risk-zone districts, {risk.population_2023.sum():,} residents ({share:.0f} % of NRW)')
risk

## 8. Dataset for Tableau

One row per district with the latest values, changes and the risk flag (saved as `data/nrw_tableau_districts.csv`), plus the full time series (`data/nrw_supply_wide.csv`).

In [ ]:
latest = wide[wide['year'] == 2024][['ags', 'kreis', 'kreis_type', 'doctors', 'gps',
                                    'general_practitioners', 'internists', 'pediatricians']]
beds = wide.pivot_table(index='ags', columns='year', values='hospital_beds')[[2015, 2022]]
beds.columns = ['beds_2015', 'beds_2022']
tableau = (latest.merge(gp[['kreis', 'gps_2015', 'change_pct', 'population_2023', 'risk_zone']], on='kreis')
                 .merge(beds, left_on='ags', right_index=True)
                 .rename(columns={'change_pct': 'gps_change_pct'}))
tableau['beds_change_pct'] = ((tableau['beds_2022'] - tableau['beds_2015']) / tableau['beds_2015'] * 100).round(1)
tableau['kreis_name'] = (tableau['kreis'].str.replace(r',.*$', '', regex=True)
                                          .str.replace('Städteregion Aachen', 'Aachen'))
tableau['state'] = 'Nordrhein-Westfalen'
tableau['country'] = 'Germany'

tableau.to_csv('nrw_tableau_districts.csv', index=False)
wide.to_csv('nrw_supply_wide.csv', index=False)
tableau.head()

## 9. Conclusions

**Where supply is lowest.** In 2024, GP supply ranges from **4.3 per 10,000 residents in Herne** to **7.2 in Leverkusen**, a gap of almost 1.7×. The weakest districts are a mix of Ruhr cities (Herne, Gelsenkirchen, Duisburg) and rural districts (Lippe, Hochsauerlandkreis, Wesel), so low GP supply is not only a rural problem.

**Urban vs rural.** GP supply does **not** differ significantly between urban and rural districts (median 5.86 vs 5.73, Mann-Whitney p = 0.89). Urban districts lost GPs faster after 2015, and by 2024 both groups average 5.8. All other indicators differ significantly (p < 0.01): cities have more doctors overall (16.3 vs 12.7), more pediatricians (6.5 vs 4.4), more internists and far more hospital beds (8.2 vs 5.3 per 1,000), because they also serve patients from surrounding districts.

**Change since 2015.** GP supply fell in **30 of 53 districts** (median change −2.4 %). Across all districts the decline is not statistically significant (Wilcoxon p = 0.11), but it is strongly concentrated: Lippe (−22 %), Krefeld (−21 %), Hochsauerlandkreis (−20 %), Mönchengladbach (−19 %) and Herne (−18 %). Hospital beds declined steadily in both urban and rural districts.

**Risk zones.** **19 districts** combine below-median GP supply with a decline since 2015. About **6.0 million people, a third of the NRW population**, live in these districts.

**Correlations.** GP supply is unrelated to hospital beds (r = 0.09): a district can be well served by hospitals and still lack GPs. Pediatricians and all doctors are strongly correlated (r = 0.90), and specialists cluster where hospitals are (doctors ↔ beds r = 0.54).

**Düsseldorf.** GP supply fell from 6.53 to 6.06 per 10,000 (−7 %) but stays above the NRW median. Hospital beds fell from 8.3 to 6.9 per 1,000 between 2015 and 2022 (−17 %).

**Recommendations**
- Prioritise GP recruitment support (e.g. funding for new practices, Weiterbildung in Allgemeinmedizin) in the 19 risk-zone districts, starting with Lippe, Herne, Hochsauerlandkreis, Krefeld and Mönchengladbach.
- Treat Ruhr cities as a target group too, not only rural areas.
- Plan GP supply separately from hospital capacity, since the two are not linked.

**Limitations**
- INKAR indicators are averages per district; differences inside a district (e.g. between city quarters) are not visible.
- Doctor counts are headcounts, not full-time equivalents, and do not include doctors' age (upcoming retirements).
- Hospital beds are counted where the hospital is, so cities appear better supplied partly because they serve neighbouring districts.